In [1]:
import warnings
warnings.filterwarnings("ignore")
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import koreanize_matplotlib
# import seaborn as sns
# import missingno as msno
# from plotnine import *
# import folium
# from wordcloud import WordCloud
import requests
# from bs4 import BeautifulSoup
# from selenium import webdriver
# import json
# from pandas import json_normalize
# import time
# %matplotlib inline
# %matplotlib notebook

mongodb를 사용하기 위해서 라이브러리를 설치하고 import 한다.

In [2]:
# !pip install pymongo
import pymongo

mongodb에 연결하고 데이터베이스 목록을 얻어온다.

In [3]:
# MongoClient() 클래스 객체로 mongodb의 url을 넘겨서 mongodb와 연결한다.
conn = pymongo.MongoClient('mongodb://localhost:27017')
# list_database_names() 메소드로 mongodb의 데이터베이스 목록을 얻어온다.
databaseList = conn.list_database_names()
print(type(databaseList))
print(databaseList)

<class 'list'>
['admin', 'config', 'local', 'starbucks', 'studydb']


starbucks 데이터베이스가 존재하나 확인한다.

In [4]:
if 'starbucks' in databaseList:
    print('starbucks 데이터베이스가 존재합니다.')
else:
    print('starbucks 데이터베이스가 존재하지 않습니다.')

starbucks 데이터베이스가 존재합니다.


In [5]:
# starbucks 데이터베이스가 존재하지 않으면 데이터베이스를 만들고 사용할 준비를 하고 존재하면 사용할 준비를 한다.
starbucks = conn['starbucks'] # test> use starbucks
# sido 컬렉션이 존재하지 않으면 컬렉션을 만들고 사용하고 존재하면 그냥 사용한다.
sido = starbucks['sido'] # starbucks> db.createCollection('sido')
# sido 컬렉션에 저장된 모든 도큐먼트를 제거한다. 삭제 조건이 없더라고 {}를 생략하면 안된다.
# insert_one() 메소드로 저장하면 기존 데이터에 추가되므로 기존 데이터를 제거한다.
sido.delete_many({}) # starbucks> db.sido.deleteMany({})

DeleteResult({'n': 16, 'ok': 1.0}, acknowledged=True)

mongodb에 저장할 도큐먼트를 딕셔너리 형태로 만든다.

In [6]:
# mongodb에 저장할 starbucks 사이트를 크롤링해서 {'sido_cd': 시도코드, 'sido_nm': 시도이름} 형태의 딕셔너리를 만들어서 리스트에 저장한다.
# response = requests.post('https://www.starbucks.co.kr/store/getSidoList.do')
# print(response)
# print(type(response.text))
# print(response.text)
# print(type(response.json()))
# print(response.json())

response = requests.post('https://www.starbucks.co.kr/store/getSidoList.do').json()
# print(response.keys())
sidoList = []
for res in response['list']:
    # print(res)
    dic = {'sido_cd': res.get('sido_cd').strip(), 'sido_nm': res['sido_nm'].strip()}
    # print(dic)
    sidoList.append(dic)
sidoList

[{'sido_cd': '01', 'sido_nm': '서울'},
 {'sido_cd': '08', 'sido_nm': '경기'},
 {'sido_cd': '02', 'sido_nm': '광주'},
 {'sido_cd': '03', 'sido_nm': '대구'},
 {'sido_cd': '04', 'sido_nm': '대전'},
 {'sido_cd': '05', 'sido_nm': '부산'},
 {'sido_cd': '06', 'sido_nm': '울산'},
 {'sido_cd': '07', 'sido_nm': '인천'},
 {'sido_cd': '09', 'sido_nm': '강원'},
 {'sido_cd': '10', 'sido_nm': '경남'},
 {'sido_cd': '11', 'sido_nm': '경북'},
 {'sido_cd': '12', 'sido_nm': '전남'},
 {'sido_cd': '13', 'sido_nm': '전북'},
 {'sido_cd': '14', 'sido_nm': '충남'},
 {'sido_cd': '15', 'sido_nm': '충북'},
 {'sido_cd': '16', 'sido_nm': '제주'},
 {'sido_cd': '17', 'sido_nm': '세종'}]

mongodb의 starbucks 데이터베이스의 sido 컬렉션에 sidoList라는 리스트에 저장된 도큐먼트를 저장한다.

In [7]:
for doc in sidoList:
    sido.insert_one(doc) # db.sido.insertOne({'sido_cd': '01', 'sido_nm': '서울'})

=============================================================================================================================================================

mongodb에 저장된 데이터를 읽어온다.

In [8]:
conn = pymongo.MongoClient('mongodb://localhost:27017')
starbucks = conn['starbucks']
sido = starbucks['sido']

In [9]:
sidoList = sido.find({}, {'_id': 0}) # starbucks> db.sido.find({}, {'_id': 0})
print(sidoList)

# mongodb에서 데이터를 find() 메소드로 얻어오면 Cursor가 처리할 데이터의 인덱스를 기억한다.
# 아래의 반복문을 실행하면 sidoList에 저장된 데이터가 소모될때마다 Cursor가 1씩 증가해서 처리할 데이터의 인덱스를 기억한다.
# for s in sidoList:
    # print(type(s))
    # print(s)
    
result = []
while sidoList:
    try:
        # print(sidoList.next())
        result.append(sidoList.next())
    except:
        break
result

[{'sido_cd': '01', 'sido_nm': '서울'},
 {'sido_cd': '08', 'sido_nm': '경기'},
 {'sido_cd': '02', 'sido_nm': '광주'},
 {'sido_cd': '03', 'sido_nm': '대구'},
 {'sido_cd': '04', 'sido_nm': '대전'},
 {'sido_cd': '05', 'sido_nm': '부산'},
 {'sido_cd': '06', 'sido_nm': '울산'},
 {'sido_cd': '07', 'sido_nm': '인천'},
 {'sido_cd': '09', 'sido_nm': '강원'},
 {'sido_cd': '10', 'sido_nm': '경남'},
 {'sido_cd': '11', 'sido_nm': '경북'},
 {'sido_cd': '12', 'sido_nm': '전남'},
 {'sido_cd': '13', 'sido_nm': '전북'},
 {'sido_cd': '14', 'sido_nm': '충남'},
 {'sido_cd': '15', 'sido_nm': '충북'},
 {'sido_cd': '16', 'sido_nm': '제주'},
 {'sido_cd': '17', 'sido_nm': '세종'}]

=============================================================================================================================================================

mongodb에 저장된 데이터를 수정한다.

python과 mongodb의 문법 차이때문에 \\$set 부분에서 SyntaxError가 발생된다. 이 문제를 해결하려면 \\$set 부분을 따옴표로 감싸주면 된다.

In [10]:
sido.update_one(
    {'sido_cd': '02'},
    {'$set':
        {'sido_nm': '전남광주통합특별시'}
    }
)
# starbucks> db.sido.updateOne(
#              {'sido_cd': '02'},
#              {$set:
#                {'sido_nm': '전남광주통합특별시'}
#              }
#            )

UpdateResult({'n': 1, 'nModified': 1, 'ok': 1.0, 'updatedExisting': True}, acknowledged=True)

=============================================================================================================================================================

mongodb에 저장된 데이터를 삭제한다.

In [11]:
sido.delete_one({'sido_cd': '12'}) # starbucks> db.sido.deleteOne({'sido_cd': '12'})

DeleteResult({'n': 1, 'ok': 1.0}, acknowledged=True)